In [1]:
import os
import torch
import numpy as np
import json
import tqdm
import torch.nn.functional as F

import datasets as datasets
import utils

<h1> 0. Key Arguments

In [2]:
datapath= "../../../data/FashionIQ/"
dataset = 'fashioniq_shirt' 
split = 'val'  

# gpt -4o 
json_path = "../new_caption/p_fashioniq_shirt_4o.json"

gpu_num = 0
clip_model_name = 'ViT-L-14' 


In [3]:
device = torch.device(f"cuda:{gpu_num}" if torch.cuda.is_available() else "cpu")

In [4]:

@torch.no_grad()
def generate_predictions( device ,  clip_model,  query_dataset  ):
    """
    Generates features predictions for the validation set of CIRCO
    """    
    ### Generate BLIP Image Captions.
    torch.cuda.empty_cache()    
    batch_size = 32
    num_workers = 8 

    query_loader = torch.utils.data.DataLoader(dataset=q_dataset, batch_size=batch_size, num_workers=num_workers, pin_memory=False, collate_fn=utils.collate_fn, shuffle=False)
    
    # if preload_dict['captions'] is None or not os.path.exists(preload_dict['captions']):
    all_captions, relative_captions = [], []
    gt_img_ids, query_ids = [], []
    target_names, reference_names = [], []
    modified_captions = []

    modified_captions_1 = []
    modified_captions_2 = []
    modified_captions_3 = []

    org_captions = []
    target_captions = []
    tokens= []
    index_features = []
    query_iterator = tqdm.tqdm(query_loader, position=0, desc='Generating image captions...')

    include_descriptions  = []
    exclude_descriptions  = []    

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
 
    json_idx = 0
    ### prediction from mod caption 
    for batch in query_iterator:
        
        if 'genecis' in dataset:
            # blip_image = batch[2].to(device)
            relative_captions.extend(batch[1])
        else:
            # blip_image = batch['blip_ref_img'].to(device)
            reference_names.extend(batch['reference_name'])
            if 'fashioniq' not in dataset:
                relative_captions.extend(batch['relative_caption'])
            else:
                rel_caps = batch['relative_captions']
                rel_caps = np.array(rel_caps).T.flatten().tolist()
                rel_caps = [f"{rel_caps[i].strip('.?, ')} and {rel_caps[i + 1].strip('.?, ')}" for i in range(0, len(rel_caps), 2)]
                relative_captions.extend( rel_caps)
                            
            if 'target_name' in batch:
                target_names.extend(batch['target_name'])
        
            gt_key = 'target'
            if 'group_members' in batch:
                gt_key = 'group_members'
            if gt_key in batch:
                gt_img_ids.extend(np.array(batch[gt_key]).T.tolist())

            query_key = 'query_id'
            if 'pair_id' in batch:
                query_key = 'pair_id'
            if query_key in batch:
                query_ids.extend(batch[query_key])

            batch_size = len(batch['reference_name'])
            batch_json = data[json_idx : json_idx + batch_size]
            json_idx += batch_size
 
            for sample in batch_json:
                response_dict = sample["resp"]
                gt_img_ids.append(sample["target"])

                try:
                    resp_dict = json.loads(response_dict) 
                    target_captions.append(resp_dict["Target Image Description"])

                    _in = ' and '.join(resp_dict["include"]) if resp_dict["include"] else ''
                    _ex = ' and '.join(resp_dict["exclude"]) if resp_dict["exclude"] else ''                    
                    include_descriptions.append( _in) 
                    exclude_descriptions.append( _ex) 
                    
                except:
                    import pdb
                    pdb.set_trace() 
                    target_captions.append("") 

    
    target_captions = np.array(target_captions)
    print( target_captions.shape ) 
    predicted_features = utils.text_encoding(device, clip_model, target_captions, batch_size=batch_size)   

    include_descriptions = np.array(include_descriptions)
    in_features = utils.text_encoding(device, clip_model, include_descriptions, batch_size=batch_size) 

    exclude_descriptions = np.array(exclude_descriptions)
    ex_features = utils.text_encoding(device, clip_model, exclude_descriptions, batch_size=batch_size)     

    relative_captions = np.array(relative_captions)
    org_features = utils.text_encoding(device, clip_model, relative_captions, batch_size=batch_size)     

    res_dict =  {
        'target_names': target_names, 
        'reference_names': reference_names,
        'targets': gt_img_ids,
        'query_ids': query_ids,
        
        'predicted_features': predicted_features, 
        'in_features': in_features, 
        'ex_features': ex_features, 
        #'in_captions': include_descriptions,
        #'ex_captions': exclude_descriptions,
        'org_features': org_features,     
        'target_captions':target_captions,
    }

    return res_dict
 

<h1> 1. Model

In [5]:
model, preprocess = utils.build_clip(clip_model_name, device=device)

/home/jylim/.conda/envs/osrcir/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jylim/.conda/envs/osrcir/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


<h1> 2. Dataset 

In [6]:
dress_type = dataset.split('_')[-1]
pool_datasets = datasets.FashionIQDataset(datapath, split, [dress_type], 'classic', preprocess )
q_dataset = datasets.FashionIQDataset(datapath, split, [dress_type], 'relative', preprocess )


FashionIQ val - ['shirt'] dataset in classic mode initialized
FashionIQ val - ['shirt'] dataset in relative mode initialized


<h1> 3. clip feature 

In [7]:
ref_img_tokens_list, ref_img_name_list , target_names, target_features  = utils.extract_image_features_(device, q_dataset ,  model , 'query') 

100%|██████████████████████████████████████████████████████████████████████| 64/64 [00:12<00:00,  5.22it/s]


In [8]:
pool_features, index_names, _ , _ = utils.extract_image_features_(device, pool_datasets ,  model , 'pool') 

100%|████████████████████████████████████████████████████████████████████| 199/199 [00:17<00:00, 11.57it/s]


In [9]:
res_dict  = generate_predictions(device, model, q_dataset,  )

Generating image captions...: 100%|████████████████████████████████████████| 64/64 [00:02<00:00, 26.19it/s]


(2038,)
text encoding 


Encoding captions...:   0%|                                                         | 0/93 [00:00<?, ?it/s]/home/jylim/.conda/envs/osrcir/lib/python3.9/site-packages/torch/nn/modules/activation.py:1160: UserWarning: Converting mask without torch.bool dtype to bool; this will negatively affect performance. Prefer to use a boolean mask directly. (Triggered internally at ../aten/src/ATen/native/transformers/attention.cpp:150.)
  return torch._native_multi_head_attention(
Encoding captions...: 100%|████████████████████████████████████████████████| 93/93 [00:02<00:00, 41.81it/s]


text encoding 


Encoding captions...: 100%|████████████████████████████████████████████████| 93/93 [00:02<00:00, 37.02it/s]


text encoding 


Encoding captions...: 100%|████████████████████████████████████████████████| 93/93 [00:02<00:00, 37.37it/s]


text encoding 


Encoding captions...: 100%|████████████████████████████████████████████████| 93/93 [00:02<00:00, 37.49it/s]


In [10]:
predicted_features = res_dict["predicted_features"]
in_features = res_dict["in_features"]
ex_features = res_dict["ex_features"]

In [11]:
org_features = res_dict["org_features"]
target_captions = res_dict["target_captions"]

<h1>  4. prediction 

In [12]:
def sph_inter(a,b,s):
    theta = torch.acos( (a*b).sum(dim=[1] )).view(a.shape[0],1)
    n1 = torch.sin(s*theta)/torch.sin(theta)*a
    n2 = torch.sin((1-s)*theta)/torch.sin(theta)*b
    return n1+n2

def combine_scores(mat, combine_mode=''):
    if combine_mode == 'alpha':
        row = alpha_list.index(combine_alpha)
        return mat[row]  # (C,)
    elif combine_mode == 'mean':
        mask = torch.isfinite(mat)
        count = mask.sum(dim=0).clamp(min=1)
        summed = torch.where(mask, mat, torch.zeros_like(mat)).sum(dim=0)
        return summed / count
    else:  # 'max'
        return mat.max(dim=0).values

In [13]:
## query 
predicted_features = torch.nn.functional.normalize(predicted_features).to(device)
ref_img_tokens_list = torch.nn.functional.normalize(ref_img_tokens_list.float()).to(device) 

## pool 
pool_features = torch.nn.functional.normalize(pool_features.float()).to(device)

## in, ex 
in_features = torch.nn.functional.normalize(in_features).to(device)
ex_features = torch.nn.functional.normalize(ex_features).to(device)
org_features = torch.nn.functional.normalize(org_features).to(device)

In [14]:

def get_conadi_list( _Topk, alpha_list, predicted_features, ref_img_tokens_list, org_features, in_features, ex_features, flag  ):
    sorted_index_names = []
    for w_feature, v_feature ,_org,  _in, _ex  in tqdm.tqdm(zip(predicted_features, ref_img_tokens_list, org_features, in_features, ex_features ), total=len(predicted_features), desc='Computing Metric.'):
    
        per_alpha_top_indices = []   
        per_alpha_top_scores_raw = []      

        for alpha in alpha_list:
            proto_pred =  sph_inter( w_feature.unsqueeze(0), v_feature , alpha)  
            similarity = (proto_pred  @ pool_features.T).squeeze()

            top_vals_raw, top_idxs_raw = torch.topk(similarity, dim=-1, k=_Topk)
            per_alpha_top_indices.append(top_idxs_raw.detach())
            per_alpha_top_scores_raw.append(top_vals_raw.detach())

        candidate_set = set()
        for top_idxs in per_alpha_top_indices:
            candidate_set.update(int(i) for i in top_idxs.tolist()) 
    
        candidate_list = sorted(candidate_set)
        candidate_tensor = torch.tensor(candidate_list, device=pool_features.device)
        C = len(candidate_list)
        A = len(per_alpha_top_indices)        
        
        sim_mat_raw = torch.full((A, C), float('-inf'), device=device)
        pos_of = {idx: pos for pos, idx in enumerate(candidate_list)}
    
        for a_idx, (top_idxs, top_vals_raw) in enumerate(
            zip(per_alpha_top_indices, per_alpha_top_scores_raw)
        ):
            for j, idx in enumerate(top_idxs.tolist()):
                pos = pos_of.get(idx, None)
                if pos is not None:
                    sim_mat_raw[a_idx, pos] = top_vals_raw[j]
        base_similarity = combine_scores(sim_mat_raw, flag)
    
    
        sim_in = None
        sim_ex = None
        if _in is not None:
            sim_in = (_in @ pool_features[candidate_tensor].T).squeeze()  
            delta_in = F.relu(base_similarity - sim_in)
        if _ex is not None:
            sim_ex = (_ex @ pool_features[candidate_tensor].T).squeeze()  
            delta_ex =  F.relu(base_similarity - sim_ex)
        
        if _org is not None:
            sim_org = (_org @ pool_features[candidate_tensor].T).squeeze()  
    

        final_score = base_similarity  +  sim_org -  delta_in  +  delta_ex

        k_final = min(_Topk, C)
        
        sorted_pos = torch.topk(final_score, dim=-1, k=k_final).indices
        final_indices = candidate_tensor[sorted_pos].cpu().numpy()
        sorted_index_name = np.array(index_names)[final_indices]
        sorted_index_names.append(sorted_index_name) 
    return sorted_index_names

In [15]:
geodesic_ratio_list =  [ 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
flag = 'max'
_Topk = 100

In [16]:
sorted_index_names = get_conadi_list( _Topk,  geodesic_ratio_list,   predicted_features, ref_img_tokens_list, org_features, in_features, ex_features, flag  )

Computing Metric.: 100%|███████████████████████████████████████████████| 2038/2038 [00:34<00:00, 59.29it/s]


In [17]:
sorted_index_names = np.array(sorted_index_names)

In [18]:
# Compute the distances
distances = (1 - predicted_features @ pool_features.T).cpu()
tmp_indices = torch.argsort(distances, dim=-1).cpu()
tmp_index_names = np.array(index_names)[tmp_indices]

In [19]:
tot_index_names = np.concatenate((sorted_index_names, tmp_index_names[:, _Topk:]), axis=1)

In [20]:
labels = torch.tensor( tot_index_names == np.repeat(np.array(target_names), len(index_names)).reshape(len(target_names), -1))

In [21]:
output_metrics = {
    'Recall@10': (torch.sum(labels[:, :10]) / len(labels)).item() * 100,
    'Recall@50': (torch.sum(labels[:, :50]) / len(labels)).item() * 100
}
print( output_metrics )

{'Recall@10': 40.87340533733368, 'Recall@50': 60.35328507423401}
